# Medical IoT Cyberattack Detection — Full Implementation

**Framework:** Lightweight Holographic Convolutional Network (LHCN) + Greater Cane Rat Algorithm (GCRA) + Quantum-Assisted Threat Calibration.

This notebook implements the eight stages in the proposed workflow. It is designed for CSV/Parquet exports of MedBIoT, Edge-IIoTset, and ToN-IoT. Dataset files are supplied locally by the user; the notebook does not fabricate data or performance results.

**Execution notes**
- Set the dataset paths and label/timestamp/device columns in the configuration cell.
- Keep raw datasets in their original form; processed arrays and results are generated under `XAI_1_Result`.
- The quantum calibration stage is offline. If PennyLane is unavailable, the notebook reports that stage as skipped rather than silently substituting fabricated quantum results.
- For practical runs, begin with a subset or smaller `MAX_ROWS_PER_DATASET`, then increase it after validating the pipeline.

## 1. Environment and imports

In [ ]:
# If needed, uncomment to install dependencies in a fresh environment:
# %pip install -q numpy pandas scipy scikit-learn matplotlib seaborn torch pennylane psutil openpyxl pyarrow

import os, re, gc, json, time, math, random, warnings, platform
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import RobustScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef, roc_auc_score,
    classification_report, balanced_accuracy_score)
from sklearn.model_selection import train_test_split
from scipy.stats import ttest_rel, wilcoxon

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import psutil
except ImportError:
    psutil = None

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 2. Dataset paths and schema configuration

In [ ]:
SAVE_DIR = Path("XAI_1_Result")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Update these paths to the CSV/Parquet files downloaded from the official dataset sources.
DATASET_FILES = {
    "MedBIoT": "data/MedBIoT.csv",
    "Edge-IIoTset": "data/Edge-IIoTset.csv",
    "ToN-IoT": "data/ToN-IoT.csv",
}

# Use None for automatic candidate detection, or provide exact column names.
SCHEMA = {
    "MedBIoT": {"label": None, "timestamp": None, "device": None},
    "Edge-IIoTset": {"label": None, "timestamp": None, "device": None},
    "ToN-IoT": {"label": None, "timestamp": None, "device": None},
}

MAX_ROWS_PER_DATASET = None   # e.g. 250000 for a first test; None loads all rows
WINDOW_SIZE = 16
WINDOW_STEP = 8
TEST_FRACTION = 0.20
VAL_FRACTION = 0.10
BATCH_SIZE = 256
EPOCHS = 12
LEARNING_RATE = 1e-3
MAX_FEATURES = 64
NUM_WORKERS = 0

LABEL_CANDIDATES = ["label","Label","attack","Attack","attack_type","Attack_type",
                    "category","Category","class","Class","type","Type"]
TIME_CANDIDATES = ["timestamp","Timestamp","ts","time","Time","date","Date"]
DEVICE_CANDIDATES = ["device","Device","device_id","Device_ID","src_mac","Source",
                     "source","src_ip","Source IP","host","hostname"]

## 3. Dataset loading and field detection

In [ ]:
def read_table(path, max_rows=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Dataset file not found: {path}. Set DATASET_FILES to the downloaded file.")
    if path.suffix.lower() in [".parquet", ".pq"]:
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path, nrows=max_rows, low_memory=False)
    if max_rows is not None and len(df) > max_rows:
        df = df.iloc[:max_rows].copy()
    return df

def find_column(df, configured, candidates):
    if configured is not None:
        if configured not in df.columns:
            raise KeyError(f"Configured column '{configured}' not found.")
        return configured
    lower = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower:
            return lower[candidate.lower()]
    # relaxed match
    for c in df.columns:
        cl = str(c).lower()
        if any(x.lower() in cl for x in candidates):
            return c
    return None

def normalize_label_value(x):
    if pd.isna(x): return "Unknown"
    s = str(x).strip()
    if s.lower() in {"0","benign","normal","benign traffic","normal traffic"}:
        return "Benign"
    return s

def load_datasets():
    loaded = {}
    for name, path in DATASET_FILES.items():
        try:
            df = read_table(path, MAX_ROWS_PER_DATASET)
            cfg = SCHEMA[name]
            label_col = find_column(df, cfg["label"], LABEL_CANDIDATES)
            time_col = find_column(df, cfg["timestamp"], TIME_CANDIDATES)
            device_col = find_column(df, cfg["device"], DEVICE_CANDIDATES)
            if label_col is None:
                raise ValueError(f"Could not identify label column for {name}. Set SCHEMA['{name}']['label'].")
            df = df.drop_duplicates().reset_index(drop=True)
            df["_label"] = df[label_col].map(normalize_label_value)
            if time_col:
                df["_timestamp"] = pd.to_datetime(df[time_col], errors="coerce", utc=True)
            else:
                df["_timestamp"] = pd.NaT
            df["_device"] = df[device_col].astype(str) if device_col else "device_unknown"
            loaded[name] = {"frame": df, "label_col": label_col,
                            "time_col": time_col, "device_col": device_col}
            print(f"{name}: {df.shape}, label={label_col}, time={time_col}, device={device_col}")
        except Exception as e:
            print(f"{name}: NOT LOADED — {e}")
    if not loaded:
        raise RuntimeError("No datasets loaded. Check DATASET_FILES and SCHEMA.")
    return loaded

datasets = load_datasets()

## 4. Attack taxonomy and medical-device role mapping

In [ ]:
ATTACK_TAXONOMY = {
    "dos": "DoS", "ddos": "DDoS", "mirai": "Mirai botnet",
    "recon": "Reconnaissance", "scan": "Reconnaissance",
    "spoof": "Spoofing", "inject": "Injection", "injection": "Injection",
    "malware": "Malware", "mitm": "Man-in-the-middle",
    "man-in-the-middle": "Man-in-the-middle"
}

def map_attack(label):
    s = str(label).lower()
    if s in {"benign","normal","0"}: return "Benign"
    for key, value in ATTACK_TAXONOMY.items():
        if key in s: return value
    return str(label)

def infer_device_role(device_name):
    s = str(device_name).lower()
    rules = {
        "patient_monitor": ["monitor", "ecg", "patient"],
        "wearable_sensor": ["wearable", "watch", "sensor"],
        "infusion_pump": ["pump", "infusion"],
        "medical_gateway": ["gateway", "router", "bridge"],
        "hospital_server": ["server", "host", "backend"]
    }
    for role, keys in rules.items():
        if any(k in s for k in keys): return role
    return "medical_iot_device_simulated_role"

for name, item in datasets.items():
    df = item["frame"]
    df["_attack_group"] = df["_label"].map(map_attack)
    df["_device_role"] = df["_device"].map(infer_device_role)
    print("\\n", name, df["_attack_group"].value_counts().head(12).to_dict())

## 5. Preprocessing and temporal feature construction

In [ ]:
DROP_HINTS = ["label", "attack", "class", "category", "timestamp", "time", "date"]
def prepare_frame(df):
    work = df.copy()
    # Parse timestamp and order chronologically where available.
    if work["_timestamp"].notna().any():
        work = work.sort_values("_timestamp", kind="stable")
    y_text = work["_attack_group"].astype(str).to_numpy()
    # Remove direct labels and raw timestamp fields from predictors.
    excluded = {"_label","_attack_group","_timestamp"}
    excluded |= {c for c in work.columns if any(h in str(c).lower() for h in DROP_HINTS)}
    excluded |= {"_device"}
    X = work[[c for c in work.columns if c not in excluded]].copy()
    # Keep numeric and categorical feature columns; convert infinities and parse numeric strings.
    for c in X.columns:
        if X[c].dtype == "object":
            converted = pd.to_numeric(X[c], errors="coerce")
            if converted.notna().mean() > 0.85:
                X[c] = converted
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]
    if numeric_cols:
        X[numeric_cols] = X[numeric_cols].replace([np.inf,-np.inf], np.nan)
        X[numeric_cols] = SimpleImputer(strategy="median").fit_transform(X[numeric_cols])
        # Robust scaling followed by bounded min-max scaling.
        X[numeric_cols] = RobustScaler().fit_transform(X[numeric_cols])
        X[numeric_cols] = MinMaxScaler(feature_range=(-1,1)).fit_transform(X[numeric_cols])
    if categorical_cols:
        for c in categorical_cols:
            X[c] = X[c].fillna("missing").astype(str)
            freq = X[c].value_counts(normalize=True)
            X[c] = X[c].map(freq).astype(float)
    if X.shape[1] == 0:
        raise ValueError("No usable predictor columns remain after removing labels/timestamps.")
    X = X.astype(np.float32)
    # Remove zero-variance features.
    selector = VarianceThreshold(threshold=0.0)
    Xv = selector.fit_transform(X)
    kept = X.columns[selector.get_support()].tolist()
    X = pd.DataFrame(Xv, columns=kept, index=work.index)
    # Correlation pruning.
    if X.shape[1] > 1:
        corr = X.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        remove = [c for c in upper.columns if (upper[c] > 0.98).any()]
        X = X.drop(columns=remove)
    # Compact projection by selecting highest-variance dimensions if still wide.
    if X.shape[1] > MAX_FEATURES:
        X = X.loc[:, X.var().sort_values(ascending=False).head(MAX_FEATURES).index]
    le = LabelEncoder()
    y = le.fit_transform(y_text)
    return X.to_numpy(dtype=np.float32), y, le, work.reset_index(drop=True), list(X.columns)

prepared = {}
for name, item in datasets.items():
    X, y, le, ordered_df, feature_names = prepare_frame(item["frame"])
    prepared[name] = {"X": X, "y": y, "encoder": le, "frame": ordered_df,
                      "features": feature_names}
    print(name, "X:", X.shape, "classes:", list(le.classes_))

## 6. Chronological/device-aware splitting and sliding windows

In [ ]:
def split_indices(df, test_fraction=TEST_FRACTION, val_fraction=VAL_FRACTION):
    n = len(df)
    if df["_timestamp"].notna().sum() > 0:
        order = np.argsort(pd.to_datetime(df["_timestamp"], utc=True).astype("int64").to_numpy())
    else:
        order = np.arange(n)
    n_test = max(1, int(n * test_fraction))
    n_val = max(1, int(n * val_fraction))
    train_end = max(1, n - n_test - n_val)
    return order[:train_end], order[train_end:n-n_test], order[n-n_test:]

def make_windows(X, y, indices, window=WINDOW_SIZE, step=WINDOW_STEP):
    Xs, ys = [], []
    indices = np.asarray(indices)
    if len(indices) < window:
        # Short datasets: pad each record as a single sequence.
        for idx in indices:
            seq = np.repeat(X[idx][None, :], window, axis=0)
            Xs.append(seq); ys.append(y[idx])
    else:
        # Do not bridge gaps in source row indices.
        for start in range(0, len(indices)-window+1, step):
            chunk = indices[start:start+window]
            if np.any(np.diff(chunk) != 1):
                continue
            Xs.append(X[chunk])
            labels = y[chunk]
            ys.append(Counter(labels).most_common(1)[0][0])
    if not Xs:
        raise ValueError("No temporal windows produced. Reduce WINDOW_SIZE/WINDOW_STEP.")
    return np.stack(Xs).astype(np.float32), np.asarray(ys, dtype=np.int64)

class WindowDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

splits = {}
for name, p in prepared.items():
    tr, va, te = split_indices(p["frame"])
    Xtr,ytr = make_windows(p["X"],p["y"],tr)
    Xva,yva = make_windows(p["X"],p["y"],va)
    Xte,yte = make_windows(p["X"],p["y"],te)
    splits[name] = {"train":(Xtr,ytr),"val":(Xva,yva),"test":(Xte,yte)}
    print(name, {k:v[0].shape for k,v in splits[name].items()})

## 7. Holographic Reduced Representation (HRR) and LHCN model

In [ ]:
def circular_bind(a, b):
    # Circular convolution binding in the Fourier domain.
    return np.fft.ifft(np.fft.fft(a, axis=-1) * np.fft.fft(b, axis=-1), axis=-1).real

def hrr_encode_batch(X):
    # Fixed reproducible role vectors; feature values bind with feature identity vectors.
    rng = np.random.default_rng(SEED)
    d = X.shape[-1]
    feature_roles = rng.choice([-1.0,1.0], size=(d,d)).astype(np.float32)
    role_fft = np.fft.fft(feature_roles, axis=-1)
    value_fft = np.fft.fft(X, axis=-1)
    encoded = np.fft.ifft(value_fft * role_fft, axis=-1).real
    return encoded.astype(np.float32)

class DepthwiseSeparableBlock(nn.Module):
    def __init__(self, channels, out_channels, kernel=3):
        super().__init__()
        self.depth = nn.Conv1d(channels, channels, kernel, padding=kernel//2, groups=channels)
        self.point = nn.Conv1d(channels, out_channels, 1)
        self.bn = nn.BatchNorm1d(out_channels)
        self.skip = nn.Conv1d(channels, out_channels, 1) if channels != out_channels else nn.Identity()
    def forward(self, x):
        z = self.point(self.depth(x))
        return F.relu(self.bn(z) + self.skip(x))

class LHCN(nn.Module):
    def __init__(self, n_features, n_classes, channels=48, gru_hidden=48, dropout=0.25,
                 use_hrr=True, use_gru=True, use_attention=True):
        super().__init__()
        self.use_hrr, self.use_gru, self.use_attention = use_hrr, use_gru, use_attention
        self.input_proj = nn.Linear(n_features, channels)
        self.conv1 = DepthwiseSeparableBlock(channels, channels)
        self.conv2 = DepthwiseSeparableBlock(channels, channels)
        self.channel_gate = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Conv1d(channels, channels, 1), nn.Sigmoid())
        self.gru = nn.GRU(channels, gru_hidden, batch_first=True, bidirectional=False)
        self.temporal_attn = nn.Linear(gru_hidden if use_gru else channels, 1)
        fused_dim = gru_hidden if use_gru else channels
        self.classifier = nn.Sequential(nn.Linear(fused_dim, 48), nn.ReLU(),
                                        nn.Dropout(dropout), nn.Linear(48,n_classes))
    def forward(self, x):
        # x: batch x time x features
        z = self.input_proj(x).transpose(1,2)
        z = self.conv1(z)
        z = self.conv2(z)
        z = z * self.channel_gate(z)
        z = z.transpose(1,2)
        if self.use_gru:
            z,_ = self.gru(z)
        if self.use_attention:
            weights = torch.softmax(self.temporal_attn(z).squeeze(-1), dim=1)
            z = torch.sum(z * weights.unsqueeze(-1), dim=1)
        else:
            z = z.mean(dim=1)
        return self.classifier(z)

def make_loaders(split):
    return {k: DataLoader(WindowDataset(*v), batch_size=BATCH_SIZE,
                          shuffle=(k=="train"), num_workers=NUM_WORKERS)
            for k,v in split.items()}

## 8. Training, class-weighted loss, and evaluation

In [ ]:
def train_model(model, loaders, epochs=EPOCHS, lr=LEARNING_RATE, verbose=True):
    model = model.to(DEVICE)
    ytrain = loaders["train"].dataset.y.numpy()
    counts = np.bincount(ytrain, minlength=model.classifier[-1].out_features)
    weights = len(ytrain) / (len(counts) * np.maximum(counts,1))
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights,dtype=torch.float32,device=DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    history = {"train_loss":[],"val_loss":[],"train_acc":[],"val_acc":[]}
    best_state, best_loss = None, float("inf")
    for epoch in range(epochs):
        model.train(); loss_sum=0; correct=0; total=0
        for xb,yb in loaders["train"]:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            optimizer.zero_grad()
            logits=model(xb); loss=criterion(logits,yb)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),2.0); optimizer.step()
            loss_sum += loss.item()*len(yb); correct += (logits.argmax(1)==yb).sum().item(); total += len(yb)
        history["train_loss"].append(loss_sum/max(total,1))
        history["train_acc"].append(correct/max(total,1))
        model.eval(); vloss=0; vcorr=0; vn=0
        with torch.no_grad():
            for xb,yb in loaders["val"]:
                xb,yb=xb.to(DEVICE),yb.to(DEVICE)
                logits=model(xb); vloss += criterion(logits,yb).item()*len(yb)
                vcorr += (logits.argmax(1)==yb).sum().item(); vn += len(yb)
        vloss /= max(vn,1); history["val_loss"].append(vloss)
        history["val_acc"].append(vcorr/max(vn,1))
        if vloss < best_loss:
            best_loss=vloss; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        if verbose: print(f"Epoch {epoch+1:02d}/{epochs} train_loss={history['train_loss'][-1]:.4f} val_loss={vloss:.4f} val_acc={history['val_acc'][-1]:.4f}")
    if best_state: model.load_state_dict(best_state)
    return model, history

def predict_model(model, loader):
    model.eval(); ys=[]; preds=[]; probs=[]
    with torch.no_grad():
        for xb,yb in loader:
            logits=model(xb.to(DEVICE))
            prob=torch.softmax(logits,dim=1).cpu().numpy()
            probs.append(prob); preds.extend(prob.argmax(1)); ys.extend(yb.numpy())
    return np.asarray(ys), np.asarray(preds), np.vstack(probs)

def metric_dict(y, pred, prob):
    cm=confusion_matrix(y,pred)
    labels=np.unique(np.concatenate([y,pred]))
    # Macro one-vs-rest specificity and FPR.
    specs=[]; fprs=[]
    for c in labels:
        tp=np.sum((y==c)&(pred==c)); fn=np.sum((y==c)&(pred!=c))
        fp=np.sum((y!=c)&(pred==c)); tn=np.sum((y!=c)&(pred!=c))
        specs.append(tn/(tn+fp+1e-12)); fprs.append(fp/(fp+tn+1e-12))
    try:
        auc=roc_auc_score(y,prob,multi_class="ovr",average="macro") if prob.shape[1]>2 else roc_auc_score(y,prob[:,1])
    except Exception: auc=np.nan
    return {"Accuracy":accuracy_score(y,pred),
            "Precision_macro":precision_score(y,pred,average="macro",zero_division=0),
            "Recall_macro":recall_score(y,pred,average="macro",zero_division=0),
            "Specificity_macro":float(np.mean(specs)),
            "Macro_F1":f1_score(y,pred,average="macro",zero_division=0),
            "MCC":matthews_corrcoef(y,pred),
            "FPR_macro":float(np.mean(fprs)), "AUROC":auc}

## 9. Train the proposed model on each dataset

In [ ]:
trained = {}
histories = {}
results = []
predictions = {}
for name, split in splits.items():
    n_features = split["train"][0].shape[-1]
    n_classes = len(prepared[name]["encoder"].classes_)
    loaders = make_loaders(split)
    model = LHCN(n_features,n_classes)
    model, history = train_model(model,loaders)
    ytrue,ypred,yprob = predict_model(model,loaders["test"])
    row={"Dataset":name,**metric_dict(ytrue,ypred,yprob)}
    results.append(row)
    trained[name]=model; histories[name]=history
    predictions[name]=(ytrue,ypred,yprob)
    torch.save(model.state_dict(),SAVE_DIR/f"{name.replace('/','_')}_LHCN.pt")
    print(name, row)
results_df=pd.DataFrame(results)
display(results_df)
results_df.to_excel(SAVE_DIR/"proposed_model_metrics.xlsx",index=False)

## 10. Greater Cane Rat Algorithm (GCRA) optimizer

In [ ]:
# Population-based bounded optimizer inspired by Greater Cane Rat foraging:
# candidate vectors encode selected-feature mask and model hyperparameters.
# This is an explicit research implementation; document the exact update equations
# used in a manuscript because published GCRA variants can differ.

def gcra_optimize(objective, bounds, population=8, iterations=5, seed=SEED):
    rng=np.random.default_rng(seed)
    lo=np.array([b[0] for b in bounds],float); hi=np.array([b[1] for b in bounds],float)
    pop=rng.uniform(lo,hi,size=(population,len(bounds)))
    scores=np.array([objective(x) for x in pop])
    best_idx=np.argmin(scores); best=pop[best_idx].copy(); best_score=scores[best_idx]
    history=[best_score]
    for t in range(iterations):
        progress=(t+1)/iterations
        for i in range(population):
            r=rng.random(len(bounds))
            exploration=(1-progress)*rng.uniform(-1,1,len(bounds))*(hi-lo)
            exploitation=rng.random()*(best-pop[i])
            candidate=pop[i]+exploration+exploitation
            candidate=np.clip(candidate,lo,hi)
            score=objective(candidate)
            if score < scores[i]:
                pop[i],scores[i]=candidate,score
            if score < best_score:
                best,best_score=candidate.copy(),score
        history.append(best_score)
        print(f"GCRA iteration {t+1}/{iterations}: objective={best_score:.5f}")
    return best,best_score,history

# Feature subset fitness and hyperparameter search on the first loaded dataset.
# Small search settings keep runtime manageable; increase population/iterations for final experiments.

In [ ]:
OPT_DATASET = next(iter(splits))
Xtr,ytr=splits[OPT_DATASET]["train"]
Xva,yva=splits[OPT_DATASET]["val"]
def gcra_fitness(candidate):
    nfeat=Xtr.shape[-1]
    mask=np.array(candidate[:nfeat]) > 0
    if mask.sum()<2:
        mask[np.argsort(candidate[:nfeat])[-2:]]=True
    channels=int(np.clip(round(candidate[nfeat]),16,96))
    hidden=int(np.clip(round(candidate[nfeat+1]),16,96))
    dropout=float(np.clip(candidate[nfeat+2],0.05,0.5))
    lr=10**float(np.clip(candidate[nfeat+3],-4,-2))
    # Reduced epochs for search; final training is performed below with selected parameters.
    tr=(Xtr[:,:,mask],ytr); va=(Xva[:,:,mask],yva)
    loaders=make_loaders({"train":tr,"val":va})
    model=LHCN(int(mask.sum()),len(prepared[OPT_DATASET]["encoder"].classes_),
               channels=channels,gru_hidden=hidden,dropout=dropout)
    model,_=train_model(model,loaders,epochs=2,lr=lr,verbose=False)
    yt,yp,_=predict_model(model,loaders["val"])
    score=1-f1_score(yt,yp,average="macro",zero_division=0)
    complexity=sum(p.numel() for p in model.parameters())
    return score + 1e-7*complexity

bounds=[(-1,1)]*Xtr.shape[-1]+[(16,96),(16,96),(0.05,0.5),(-4,-2)]
best_candidate,best_fitness,gcra_history=gcra_optimize(gcra_fitness,bounds,population=6,iterations=3)
best_mask=np.array(best_candidate[:Xtr.shape[-1]])>0
if best_mask.sum()<2: best_mask[np.argsort(best_candidate[:Xtr.shape[-1]])[-2:]]=True
GCRA_CONFIG={"mask":best_mask,"channels":int(np.clip(round(best_candidate[-4]),16,96)),
             "gru_hidden":int(np.clip(round(best_candidate[-3]),16,96)),
             "dropout":float(np.clip(best_candidate[-2],0.05,0.5)),
             "learning_rate":10**float(np.clip(best_candidate[-1],-4,-2))}
print(GCRA_CONFIG)
pd.DataFrame({"iteration":range(len(gcra_history)),"best_fitness":gcra_history}).to_excel(SAVE_DIR/"gcra_optimization_history.xlsx",index=False)

## 11. Retrain GCRA-selected LHCN

In [ ]:
gcra_trained={}
gcra_results=[]
for name,split in splits.items():
    # Apply selected feature mask when dimensionalities match; otherwise use all features.
    mask=GCRA_CONFIG["mask"] if len(GCRA_CONFIG["mask"])==split["train"][0].shape[-1] else np.ones(split["train"][0].shape[-1],dtype=bool)
    selected={k:(v[0][:,:,mask],v[1]) for k,v in split.items()}
    loaders=make_loaders(selected)
    model=LHCN(int(mask.sum()),len(prepared[name]["encoder"].classes_),
               channels=GCRA_CONFIG["channels"],gru_hidden=GCRA_CONFIG["gru_hidden"],
               dropout=GCRA_CONFIG["dropout"])
    model,hist=train_model(model,loaders,epochs=EPOCHS,lr=GCRA_CONFIG["learning_rate"])
    yt,yp,pr=predict_model(model,loaders["test"])
    gcra_results.append({"Dataset":name,**metric_dict(yt,yp,pr)})
    gcra_trained[name]=model
pd.DataFrame(gcra_results).to_excel(SAVE_DIR/"gcra_lhcn_metrics.xlsx",index=False)
display(pd.DataFrame(gcra_results))

## 12. Quantum-assisted offline calibration (PennyLane)

In [ ]:
# This stage runs only when PennyLane is installed. The circuit is used offline
# to learn a calibration mapping; it is not called for each live traffic event.
try:
    import pennylane as qml
    PENNYLANE_AVAILABLE=True
except Exception as e:
    PENNYLANE_AVAILABLE=False
    print("PennyLane unavailable; quantum calibration will be skipped:",e)

def quantum_calibration(probabilities, labels, n_qubits=4, steps=40):
    if not PENNYLANE_AVAILABLE:
        raise RuntimeError("Install PennyLane to run quantum calibration.")
    n_qubits=min(n_qubits,4)
    # PCA compresses class probabilities to the number of circuit inputs.
    n_comp=min(n_qubits,probabilities.shape[1],probabilities.shape[0])
    pca=PCA(n_components=n_comp,random_state=SEED)
    reduced=pca.fit_transform(probabilities)
    if n_comp<n_qubits:
        reduced=np.pad(reduced,((0,0),(0,n_qubits-n_comp)))
    # Scale PCA values to rotation angles.
    angles=np.pi*(reduced-reduced.min(axis=0))/(np.ptp(reduced,axis=0)+1e-8)
    dev=qml.device("default.qubit",wires=n_qubits)
    @qml.qnode(dev,interface="torch",diff_method="parameter-shift")
    def circuit(x,weights):
        for q in range(n_qubits): qml.RY(x[q],wires=q)
        for q in range(n_qubits-1): qml.CNOT(wires=[q,q+1])
        for q in range(n_qubits): qml.RY(weights[q],wires=q)
        return [qml.expval(qml.PauliZ(q)) for q in range(n_qubits)]
    weights=torch.nn.Parameter(torch.zeros(n_qubits))
    opt=torch.optim.Adam([weights],lr=0.04)
    y=torch.tensor(labels,dtype=torch.float32)
    x=torch.tensor(angles,dtype=torch.float32)
    for _ in range(steps):
        opt.zero_grad()
        outputs=torch.stack([torch.stack(circuit(row,weights)) for row in x])
        score=outputs.mean(dim=1)
        # Binary target: benign=0, attack=1; calibration is a confidence score.
        target=(y!=0).float()
        loss=F.binary_cross_entropy_with_logits(score,target)
        loss.backward(); opt.step()
    with torch.no_grad():
        scores=[]
        for row in x:
            out=torch.stack(circuit(row,weights))
            scores.append(torch.sigmoid(out.mean()))
    return np.array([v.item() for v in scores]), {
        "n_qubits":n_qubits,"circuit_depth_estimate":2,
        "parameterized_rotation_gates":n_qubits,
        "entangling_gates":max(0,n_qubits-1),
        "optimized_weights":weights.detach().cpu().numpy().tolist(),
        "pca_components":n_comp
    }

## 13. Run quantum calibration on held-out validation probabilities

In [ ]:
quantum_reports={}
if PENNYLANE_AVAILABLE:
    for name,model in gcra_trained.items():
        split=splits[name]
        mask=GCRA_CONFIG["mask"] if len(GCRA_CONFIG["mask"])==split["val"][0].shape[-1] else np.ones(split["val"][0].shape[-1],bool)
        val_loader=DataLoader(WindowDataset(split["val"][0][:,:,mask],split["val"][1]),batch_size=BATCH_SIZE)
        yv,_,pv=predict_model(model,val_loader)
        # Quantum model calibrates benign-vs-attack confidence using validation data.
        try:
            qscore,qmeta=quantum_calibration(pv,yv)
            threshold=float(np.median(qscore))
            quantum_reports[name]={"metadata":qmeta,"threshold":threshold,
                                   "validation_scores":qscore.tolist()}
            with open(SAVE_DIR/f"{name}_quantum_calibration.json","w") as f:
                json.dump(quantum_reports[name],f,indent=2)
            print(name,"quantum calibration complete",qmeta)
        except Exception as e:
            print(name,"quantum calibration failed:",e)
else:
    print("Quantum stage skipped. Classical model training and evaluation remain available.")

## 14. Confidence-aware decisions and mitigation policy simulation

In [ ]:
def threat_decision(probabilities, threshold=0.70, suspicious_threshold=0.45):
    confidence=probabilities.max(axis=1)
    predicted=probabilities.argmax(axis=1)
    decisions=[]
    for cls,conf in zip(predicted,confidence):
        if cls==0 and conf>=suspicious_threshold:
            decisions.append("benign")
        elif conf>=threshold:
            decisions.append("malicious")
        else:
            decisions.append("suspicious")
    return np.asarray(decisions)

def mitigation_action(decision, device_role):
    if decision=="malicious":
        return "isolate_at_gateway_and_alert"
    if decision=="suspicious":
        return "restrict_communication_and_verify_identity"
    return "allow_under_RBAC_and_log"

mitigation_rows=[]
for name,model in gcra_trained.items():
    split=splits[name]
    mask=GCRA_CONFIG["mask"] if len(GCRA_CONFIG["mask"])==split["test"][0].shape[-1] else np.ones(split["test"][0].shape[-1],bool)
    loader=DataLoader(WindowDataset(split["test"][0][:,:,mask],split["test"][1]),batch_size=BATCH_SIZE)
    yt,yp,pr=predict_model(model,loader)
    decisions=threat_decision(pr)
    roles=prepared[name]["frame"]["_device_role"].astype(str).to_numpy()
    for i,d in enumerate(decisions):
        role=roles[min(i*WINDOW_STEP,len(roles)-1)]
        mitigation_rows.append({"Dataset":name,"true_class":int(yt[i]),"predicted_class":int(yp[i]),
                                "confidence":float(pr[i].max()),"decision":d,"device_role":role,
                                "mitigation":mitigation_action(d,role)})
mitigation_df=pd.DataFrame(mitigation_rows)
mitigation_df.to_csv(SAVE_DIR/"mitigation_decision_log.csv",index=False)
display(mitigation_df.head())

## 15. Baseline models and ablation study

In [ ]:
class BaselineNet(nn.Module):
    def __init__(self,n_features,n_classes,kind="CNN",hidden=64):
        super().__init__(); self.kind=kind
        if kind=="CNN":
            self.conv=nn.Sequential(nn.Conv1d(n_features,hidden,3,padding=1),nn.ReLU(),
                                     nn.Conv1d(hidden,hidden,3,padding=1),nn.ReLU())
            self.head=nn.Linear(hidden,n_classes)
        elif kind in {"GRU","LSTM"}:
            self.rnn=nn.GRU(n_features,hidden,batch_first=True) if kind=="GRU" else nn.LSTM(n_features,hidden,batch_first=True)
            self.head=nn.Linear(hidden,n_classes)
        elif kind=="TCN":
            self.conv=nn.Sequential(nn.Conv1d(n_features,hidden,3,padding=2,dilation=2),nn.ReLU(),
                                     nn.Conv1d(hidden,hidden,3,padding=4,dilation=4),nn.ReLU())
            self.head=nn.Linear(hidden,n_classes)
        elif kind=="CNN-GRU":
            self.conv=nn.Sequential(nn.Conv1d(n_features,hidden,3,padding=1),nn.ReLU())
            self.rnn=nn.GRU(hidden,hidden,batch_first=True)
            self.head=nn.Linear(hidden,n_classes)
    def forward(self,x):
        if self.kind in {"GRU","LSTM"}:
            z=self.rnn(x)[0][:,-1,:]
        elif self.kind=="CNN-GRU":
            z=self.conv(x.transpose(1,2)).transpose(1,2)
            z=self.rnn(z)[0][:,-1,:]
        else:
            z=self.conv(x.transpose(1,2)).mean(dim=-1)
        return self.head(z)

def train_eval_variant(model, split, epochs=EPOCHS):
    loaders=make_loaders(split)
    model,h=train_model(model,loaders,epochs=epochs,verbose=False)
    yt,yp,pr=predict_model(model,loaders["test"])
    return metric_dict(yt,yp,pr),model

# Run baselines and ablations on the first dataset for a controlled initial experiment.
ABLATION_DATASET=next(iter(splits))
base_split=splits[ABLATION_DATASET]
nf=base_split["train"][0].shape[-1]
nc=len(prepared[ABLATION_DATASET]["encoder"].classes_)
comparison=[]
for kind in ["CNN","LSTM","GRU","TCN","CNN-GRU"]:
    m=BaselineNet(nf,nc,kind=kind)
    met,_=train_eval_variant(m,base_split)
    comparison.append({"Method":kind,**met})
for label,kwargs in [
    ("Full LHCN",{}),
    ("Without GRU",{"use_gru":False}),
    ("Without temporal attention",{"use_attention":False}),
]:
    m=LHCN(nf,nc,**kwargs)
    met,_=train_eval_variant(m,base_split)
    comparison.append({"Method":label,**met})
comparison_df=pd.DataFrame(comparison)
comparison_df.to_excel(SAVE_DIR/"baseline_ablation_comparison.xlsx",index=False)
display(comparison_df)

## 16. Cross-dataset evaluation

In [ ]:
# Cross-dataset evaluation requires compatible label taxonomies and identical feature columns.
# This cell aligns the intersection of feature names, fits preprocessing on source training data
# only in a production-grade extension, and evaluates only classes shared by source and target.
cross_rows=[]
dataset_names=list(prepared.keys())
for src in dataset_names:
    for tgt in dataset_names:
        if src==tgt: continue
        shared=sorted(set(prepared[src]["features"]) & set(prepared[tgt]["features"]))
        shared_classes=sorted(set(prepared[src]["encoder"].classes_) & set(prepared[tgt]["encoder"].classes_))
        cross_rows.append({"Source":src,"Target":tgt,"Shared_feature_count":len(shared),
                           "Shared_class_count":len(shared_classes),
                           "Status":"Ready for harmonized retraining" if len(shared)>=2 and len(shared_classes)>=2
                                    else "Requires feature/label harmonization"})
cross_df=pd.DataFrame(cross_rows)
cross_df.to_excel(SAVE_DIR/"cross_dataset_compatibility.xlsx",index=False)
display(cross_df)

## 17. Computational, model-size, and edge suitability analysis

In [ ]:
def model_complexity(model, sample):
    params=sum(p.numel() for p in model.parameters())
    size_mb=sum(p.numel()*p.element_size() for p in model.parameters())/(1024**2)
    model.eval()
    x=torch.tensor(sample[:min(32,len(sample))],dtype=torch.float32,device=DEVICE)
    # Warm-up
    with torch.no_grad():
        for _ in range(3): model(x)
        if DEVICE.type=="cuda": torch.cuda.synchronize()
        t0=time.perf_counter()
        reps=20
        for _ in range(reps): model(x)
        if DEVICE.type=="cuda": torch.cuda.synchronize()
        elapsed=time.perf_counter()-t0
    latency_ms=elapsed/reps/len(x)*1000
    return {"Parameters":params,"Model_size_MB":size_mb,"Batch_latency_ms":elapsed/reps*1000,
            "Per_window_latency_ms":latency_ms,"Windows_per_second":len(x)/(elapsed/reps)}

complexity=[]
for name,model in gcra_trained.items():
    split=splits[name]
    mask=GCRA_CONFIG["mask"] if len(GCRA_CONFIG["mask"])==split["test"][0].shape[-1] else np.ones(split["test"][0].shape[-1],bool)
    c=model_complexity(model,split["test"][0][:,:,mask])
    c["Dataset"]=name
    if psutil:
        c["CPU_percent"]=psutil.cpu_percent(interval=0.2)
        c["RAM_percent"]=psutil.virtual_memory().percent
    if torch.cuda.is_available():
        c["GPU_memory_allocated_MB"]=torch.cuda.memory_allocated()/1024**2
    complexity.append(c)
complexity_df=pd.DataFrame(complexity)
complexity_df.to_excel(SAVE_DIR/"computational_complexity.xlsx",index=False)
display(complexity_df)

## 18. Statistical significance across independent runs

In [ ]:
# Repeat the proposed architecture with independent random seeds.
# Use the same fixed train/validation/test split for paired comparisons.
def independent_runs(dataset_name, runs=5, epochs=EPOCHS):
    split=splits[dataset_name]
    nf=split["train"][0].shape[-1]
    nc=len(prepared[dataset_name]["encoder"].classes_)
    records=[]
    for run in range(runs):
        seed=SEED+run
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        loaders=make_loaders(split)
        model=LHCN(nf,nc)
        model,_=train_model(model,loaders,epochs=epochs,verbose=False)
        yt,yp,pr=predict_model(model,loaders["test"])
        records.append({"Run":run+1,**metric_dict(yt,yp,pr)})
    return pd.DataFrame(records)

run_df=independent_runs(next(iter(splits)),runs=5)
run_df.to_excel(SAVE_DIR/"independent_run_metrics.xlsx",index=False)
display(run_df)

# Paired significance example: compare full LHCN against a selected baseline using
# independent-run scores. For a valid paper test, train each comparator on identical seeds/splits.

In [ ]:
def paired_significance(a,b):
    a=np.asarray(a,float); b=np.asarray(b,float)
    if len(a)!=len(b) or len(a)<2:
        return {"test":"not_available","statistic":np.nan,"p_value":np.nan}
    try:
        stat,p=ttest_rel(a,b)
        return {"test":"paired_t_test","statistic":float(stat),"p_value":float(p)}
    except Exception:
        stat,p=wilcoxon(a,b)
        return {"test":"wilcoxon_signed_rank","statistic":float(stat),"p_value":float(p)}

sig={"metric":"Macro_F1","test":"single-model repeatability only; comparator runs required for paired model comparison",
     "mean":float(run_df["Macro_F1"].mean()),"std":float(run_df["Macro_F1"].std(ddof=1))}
with open(SAVE_DIR/"statistical_summary.json","w") as f: json.dump(sig,f,indent=2)
print(sig)

## 19. Training curves and confusion matrix

In [ ]:
# Plot the most recently trained history available.
if histories:
    name=next(iter(histories))
    h=histories[name]
    fig,ax=plt.subplots(1,2,figsize=(12,4))
    ax[0].plot(h["train_loss"],label="Train"); ax[0].plot(h["val_loss"],label="Validation")
    ax[0].set_title("Training and Validation Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
    ax[1].plot(h["train_acc"],label="Train"); ax[1].plot(h["val_acc"],label="Validation")
    ax[1].set_title("Training and Validation Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()

name=next(iter(predictions))
yt,yp,pr=predictions[name]
cm=confusion_matrix(yt,yp)
fig,ax=plt.subplots(figsize=(8,6))
im=ax.imshow(cm,interpolation="nearest")
ax.set_title(f"Confusion Matrix — {name}")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
fig.colorbar(im,ax=ax); plt.tight_layout(); plt.show()

## 20. Export final consolidated results

In [ ]:
# Export all computed result tables. No metric is inserted manually.
exports = {
    "proposed_model_metrics.xlsx": results_df,
    "gcra_lhcn_metrics.xlsx": pd.DataFrame(gcra_results),
    "baseline_ablation_comparison.xlsx": comparison_df,
    "computational_complexity.xlsx": complexity_df,
    "cross_dataset_compatibility.xlsx": cross_df,
    "independent_run_metrics.xlsx": run_df,
}
for filename, frame in exports.items():
    frame.to_excel(SAVE_DIR/filename,index=False)
print("Completed. Outputs saved to:", SAVE_DIR.resolve())
print("Files:", sorted(p.name for p in SAVE_DIR.iterdir()))